In [ ]:
import torch

from onomatron.data import Vocab, load_names, make_sequences, split_names
from onomatron.models.mamba import MambaName
from onomatron.utils import lr_range_test, plot_history, plot_lr_range, train

# Apple Silicon: use the integrated GPU via MPS when available (much faster
# than CPU for this model); fall back to CPU otherwise.
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

## Load names, split train/val/test, build vocab

In [ ]:
names = load_names("../data/processed/names.txt")
train_names, val_names, test_names = split_names(names, seed=42)
vocab = Vocab(train_names)  # vocab from train only: no leakage from val/test
print(
    f"{len(names)} names -> {len(train_names)} train / {len(val_names)} val"
    f" / {len(test_names)} test, vocab of {len(vocab)} chars"
)

## Encode names as full sequences (x, y)

Same as the RNN notebooks: each name is `[SENTINEL] + chars + [SENTINEL]`,
right-padded to the max length T. `x = seq[:-1]`, `y = seq[1:]`; padding
targets are -100 and ignored in the loss.

In [ ]:
X, Y = make_sequences(train_names, vocab)
X_val, Y_val = make_sequences(val_names, vocab)
X_test, Y_test = make_sequences(test_names, vocab)
# Move everything to the device once: the training loop indexes these tensors
X, Y = X.to(device), Y.to(device)
X_val, Y_val = X_val.to(device), Y_val.to(device)
X_test, Y_test = X_test.to(device), Y_test.to(device)
print(
    f"{X.shape[0]} train / {X_val.shape[0]} val / {X_test.shape[0]} test"
    f" sequences of length {X.shape[1]}"
)

## Build Mamba (Gu & Dao 2023)

`nn.Embedding` -> Mamba blocks -> LayerNorm -> Linear. Each block: causal
depthwise conv, then a **selective SSM** scan where A, B, dt depend on the
input x_t (the selection mechanism): h_t = exp(dt·A) h_{t-1} + dt·B x_t.
Training uses the vectorized log-space cumsum form; `sample()` chains the
recurrent state one char at a time (O(1) per char).

In [ ]:
n_emb, n_layer, d_state = 64, 2, 16
seed = 42

model = MambaName(vocab, n_emb, n_layer, d_state, seed).to(device)
print(f"Mamba model has {model.n_parameters} parameters.")

## Find a maximum learning rate

Lower lr range than the RNNs (transformer-like sensitivity, high lr explodes):

In [ ]:
# Pick lr (minibatch, can take a while)
lrs = (10 ** torch.linspace(-4, -0.5, 100)).tolist()
losses = lr_range_test(
    lambda s: MambaName(vocab, n_emb, n_layer, d_state, seed=s).to(device),
    X,
    Y,
    lrs,
    batch_size=64,
)
best_lr = plot_lr_range(lrs, losses) / 10
print(f"Maximum learning rate found is {best_lr}")

## Train on train, watch val

In [ ]:
n_steps = 15000
batch_size = 64
lr = best_lr

model = MambaName(vocab, n_emb, n_layer, d_state, seed).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=best_lr, weight_decay=0.1)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=n_steps)

history = train(
    model,
    X,
    Y,
    opt,
    steps=n_steps,
    batch_size=batch_size,
    scheduler=sched,
    seed=42,
    x_val=X_val,
    y_val=Y_val,
    eval_every=100,
    clip_grad=1.0,
    patience=20,
)
plot_history(history, smooth=20)
print(
    f"Final train loss {history.train_losses[-1]:.4f}"
    f" | val NLL {history.val[-1][1]:.4f}"
    f" | best val NLL {history.best[1]:.4f} at step {history.best[0]}"
)

## Sample from the model (min 3 chars)

In [ ]:
model.cpu()  # sampling runs one char at a time
model.sample(20)

## Final evaluation on test

In [ ]:
model.to(device)
print(
    f"Mamba: train NLL {model.nll(X, Y):.4f}"
    f" | val NLL {model.nll(X_val, Y_val):.4f}"
    f" | test NLL {model.nll(X_test, Y_test):.4f}"
)